[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/16_production_serving/16_proyecto_cinematch_microservicio.ipynb)

# Proyecto 16 · Desplegar CineMatch como microservicio de recomendación

**Nivel:** 🔴 Experto · **Duración:** 4–6 h · **GPU:** no necesaria · **Unidades Colab:** 1–3 (CPU)

## 🏢 Contexto de negocio

Eres *ML platform engineer* en **CineMatch**. El equipo de producto lanza la nueva home el mes que viene y el modelo
de los módulos anteriores vive todavía en un notebook. Tu misión: convertirlo en un **servicio** que cumpla el contrato
acordado con el equipo de backend:

- `GET /recommend/{user_id}?k=10` → 10 títulos, **p99 ≤ 80 ms** medido desde el cliente con 8 usuarios concurrentes (sin caché), 1 réplica.
- **Nunca** un error para usuarios nuevos: *fallback* de popularidad.
- `POST /event` actualiza el historial del usuario (nearline) e **invalida** su caché.
- `GET /metrics` en formato Prometheus para que SRE monte alertas (lo usarás en el módulo 17).
- Features **point-in-time correctas** en entrenamiento y **paridad 100 %** entre las features que ve el modelo en entrenamiento y en serving.

## 📦 Dataset

**MovieLens-100K** (GroupLens, https://grouplens.org/datasets/movielens/) — 100 000 valoraciones de 943 usuarios sobre
1 682 películas (1997–98) con géneros. Con `SCALE = "full"` usarás **MovieLens-1M** (1M valoraciones, 6 040 usuarios).
Si no hay red, el notebook genera datos sintéticos con el mismo esquema (no te quedas bloqueado).

## ✅ Entregables y rúbrica (100 puntos)

| # | Entregable | Criterio objetivo | Puntos |
|---|---|---|---|
| E1 | `pit_item_features` | test automático: ninguna feature con `event_timestamp` > instante del ejemplo; gap AUC offline/producción < 0,01 | 20 |
| E2 | Ranker LambdaMART (`LGBMRanker`) | NDCG@10 en test temporal ≥ **1,10 ×** el NDCG@10 del retrieval solo | 20 |
| E3 | Servicio FastAPI | `/health`, `/recommend`, `/event`, `/metrics`; fallback para usuarios nuevos; invalidación de caché comprobada | 25 |
| E4 | Test de carga | p99 ≤ 80 ms con concurrencia 8 sin caché (CPU de Colab) | 15 |
| E5 | Paridad de features | features **logueadas por el servicio** == features del builder offline en ≥ 99,9 % de las celdas | 10 |
| E6 | Decisiones | 5–10 líneas: qué iría a Triton/Redis/Feast/Kafka en `reference_stack/` y por qué | 10 |

> La solución de referencia está al final. Las celdas de comprobación capturan `NotImplementedError` para que puedas
> ejecutar el notebook de arriba a abajo mientras trabajas.

In [ ]:
!pip install -q fastapi uvicorn faiss-cpu lightgbm fakeredis redis requests pyarrow

In [ ]:
import os, io, zipfile, urllib.request, time, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

warnings.filterwarnings("ignore")
seed = 42
random.seed(seed); np.random.seed(seed)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DATA_DIR, ART_DIR = "data", "artifacts"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(ART_DIR, exist_ok=True)
ML100K_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
GENRES = ["unknown", "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
          "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
          "Romance", "Sci-Fi", "Thriller", "War", "Western"]


def synthetic_movielens(n_users=943, n_items=1682, n_ratings=100_000, seed=42):
    """Fallback sin red: imita el esquema de ML-100K (gustos por género + popularidad Zipf)."""
    rng = np.random.default_rng(seed)
    item_genres = (rng.random((n_items, len(GENRES))) < 0.12).astype(int)
    item_genres[np.arange(n_items), rng.integers(1, len(GENRES), n_items)] = 1
    user_taste = rng.dirichlet(np.ones(len(GENRES)) * 0.3, n_users)
    pop = 1.0 / np.arange(1, n_items + 1) ** 0.9
    pop = pop[rng.permutation(n_items)]
    rows, t0 = [], 874_724_710  # sept-1997, como ML-100K
    per_user = rng.multinomial(n_ratings, rng.dirichlet(np.ones(n_users)))
    for u in range(n_users):
        k = max(per_user[u], 20)
        p = pop * (item_genres @ user_taste[u] + 0.02)
        p /= p.sum()
        items = rng.choice(n_items, size=min(k, n_items // 2), replace=False, p=p)
        aff = item_genres[items] @ user_taste[u]
        r = np.clip(np.round(2.5 + 6 * aff + rng.normal(0, 0.8, len(items))), 1, 5)
        ts = t0 + np.sort(rng.integers(0, 215 * 86400, len(items)))
        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": items + 1, "rating": r.astype(int), "ts": ts}))
    ratings = pd.concat(rows, ignore_index=True)
    items_df = pd.DataFrame(item_genres, columns=GENRES)
    items_df.insert(0, "title", [f"Película sintética {i + 1}" for i in range(n_items)])
    items_df.insert(0, "item_id", np.arange(1, n_items + 1))
    return ratings, items_df, "Sintético (esquema ML-100K)"


def load_ml100k():
    try:
        path = os.path.join(DATA_DIR, "ml-100k.zip")
        if not os.path.exists(path):
            urllib.request.urlretrieve(ML100K_URL, path)
        with zipfile.ZipFile(path) as z:
            ratings = pd.read_csv(z.open("ml-100k/u.data"), sep="\t",
                                  names=["user_id", "item_id", "rating", "ts"])
            raw = pd.read_csv(z.open("ml-100k/u.item"), sep="|", encoding="latin-1", header=None)
        items_df = raw[[0, 1] + list(range(5, 24))].copy()
        items_df.columns = ["item_id", "title"] + GENRES
        return ratings, items_df, "MovieLens-100K"
    except Exception as e:  # sin red / URL caída → nunca bloqueamos el notebook
        print("⚠️ No se pudo descargar ML-100K, uso datos sintéticos:", repr(e)[:120])
        return synthetic_movielens()


ratings, items_df, DATASET = load_ml100k()
ratings = ratings.sort_values("ts").reset_index(drop=True)
print(DATASET, ratings.shape, "usuarios:", ratings.user_id.nunique(), "ítems:", ratings.item_id.nunique())

In [ ]:
# SCALE = "small" → MovieLens-100K (minutos en CPU) · "full" → MovieLens-1M (más realista; sigue cabiendo en CPU)
SCALE = "small"
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"


def load_ml1m():
    path = os.path.join(DATA_DIR, "ml-1m.zip")
    if not os.path.exists(path):
        urllib.request.urlretrieve(ML1M_URL, path)
    with zipfile.ZipFile(path) as z:
        r = pd.read_csv(z.open("ml-1m/ratings.dat"), sep="::", engine="python", names=["user_id", "item_id", "rating", "ts"])
        mv = pd.read_csv(z.open("ml-1m/movies.dat"), sep="::", engine="python", names=["item_id", "title", "genres"],
                         encoding="latin-1")
    g = mv.genres.str.replace("Children's", "Children").str.get_dummies("|")
    items = pd.concat([mv[["item_id", "title"]], g.reindex(columns=GENRES, fill_value=0)], axis=1)
    return r, items, "MovieLens-1M"


if SCALE == "full":
    try:
        ratings, items_df, DATASET = load_ml1m()
        ratings = ratings.sort_values("ts").reset_index(drop=True)
    except Exception as e:
        print("⚠️ ML-1M no disponible, sigo con", DATASET, repr(e)[:100])
print(DATASET, ratings.shape)

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

N_USERS, N_ITEMS = int(ratings.user_id.max()) + 1, int(items_df.item_id.max()) + 1
GENRE_MAT = np.zeros((N_ITEMS, len(GENRES)), dtype=np.float32)
GENRE_MAT[items_df.item_id.values] = items_df[GENRES].values

# Split temporal global: 70 % (embeddings) · 10 % (etiquetas del ranker) · 20 % (test)
t_a, t_b = ratings.ts.quantile([0.7, 0.8]).values
train_a = ratings[ratings.ts < t_a]
train_b = ratings[(ratings.ts >= t_a) & (ratings.ts < t_b)]
test = ratings[ratings.ts >= t_b]


def item_embeddings(df, dim=64):
    """PureSVD sobre la matriz implícita (Cremonesi et al. 2010): baseline de retrieval fuerte y barato."""
    m = csr_matrix((np.ones(len(df), dtype=np.float32), (df.user_id, df.item_id)), shape=(N_USERS, N_ITEMS))
    _, s, vt = svds(m, k=dim, random_state=seed)
    emb = (vt.T * np.sqrt(s)).astype(np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-8
    return emb


def user_vector(hist_items, emb, max_len=50):
    """'Torre de usuario' sin parámetros: media de los embeddings de su historial reciente (cf. YouTube DNN 2016)."""
    if len(hist_items) == 0:
        return None
    v = emb[np.asarray(hist_items[-max_len:])].mean(0)
    return (v / (np.linalg.norm(v) + 1e-8)).astype(np.float32)


ITEM_EMB = item_embeddings(train_a)
hist_a = train_a.groupby("user_id").item_id.apply(list).to_dict()
print("Embeddings de ítem:", ITEM_EMB.shape, "| usuarios con historial:", len(hist_a))

In [ ]:
import faiss, lightgbm as lgb, requests, subprocess, sys
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor

INDEX = faiss.IndexHNSWFlat(ITEM_EMB.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
INDEX.hnsw.efSearch = 64
INDEX.add(ITEM_EMB)


def retrieve(hist, k=100):
    v = user_vector(hist, ITEM_EMB)
    if v is None:
        return np.array([], dtype=int), np.array([], dtype=np.float32)
    s, ids = INDEX.search(v[None], k + len(hist))
    seen = set(hist)
    mask = np.array([(i not in seen) and i > 0 for i in ids[0]])
    return ids[0][mask][:k], s[0][mask][:k]


def ndcg_at_k(ranked, relevant, k=10):
    dcg = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in relevant)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(k, len(relevant))))
    return dcg / idcg if idcg else 0.0


# Snapshots diarios de features de ítem, publicados al día siguiente (batch nocturno)
ev = ratings[["item_id", "rating", "ts"]].assign(day=lambda d: d.ts // 86400)
daily = ev.groupby(["item_id", "day"]).agg(n=("rating", "size"), s=("rating", "sum")).reset_index().sort_values(["item_id", "day"])
daily["pop_cum"] = daily.groupby("item_id").n.cumsum()
daily["rating_mean_cum"] = daily.groupby("item_id").s.cumsum() / daily.pop_cum
daily["event_timestamp"] = pd.to_datetime((daily.day + 1) * 86400, unit="s", utc=True)
ITEM_STATS = daily[["item_id", "event_timestamp", "pop_cum", "rating_mean_cum"]].reset_index(drop=True)
RANK_FEATS = ["retr_score", "retr_rank", "pop_cum", "rating_mean_cum", "genre_affinity", "user_n", "user_mean"]
print("Snapshots de ítem:", ITEM_STATS.shape)

## Paso 1 · Features *point-in-time* (E1)

Implementa `pit_item_features(examples, item_stats)`: para cada fila de `examples` (`item_id`, `event_timestamp`)
añade `pop_cum` y `rating_mean_cum` del **último snapshot con `event_timestamp` ≤ instante del ejemplo**. Valores por
defecto si no hay snapshot: `pop_cum = 0`, `rating_mean_cum = 3.0`. Añade también la columna `feature_ts` con el
timestamp del snapshot usado (para el test).

💡 Pista: `pd.merge_asof(..., on="event_timestamp", by="item_id", direction="backward")`, ambos DataFrames ordenados por la clave temporal.

In [ ]:
def pit_item_features(examples: pd.DataFrame, item_stats: pd.DataFrame) -> pd.DataFrame:
    # TODO: point-in-time join + defaults + columna feature_ts
    raise NotImplementedError


def make_examples(df_pos, n_neg=4, seed=seed):
    r_ = np.random.default_rng(seed)
    pos = df_pos[["user_id", "item_id", "ts"]].assign(label=1)
    neg = pos.loc[pos.index.repeat(n_neg)].copy()
    neg["item_id"] = r_.integers(1, N_ITEMS, len(neg)); neg["label"] = 0
    out = pd.concat([pos, neg], ignore_index=True)
    out["event_timestamp"] = pd.to_datetime(out.ts, unit="s", utc=True)
    return out.sort_values("event_timestamp").reset_index(drop=True)


def check_E1():
    ex = make_examples(train_b)
    f = pit_item_features(ex, ITEM_STATS)
    ok_ts = (f.feature_ts.isna() | (f.feature_ts <= f.event_timestamp)).all()
    tr = pit_item_features(make_examples(train_a.sample(frac=0.5, random_state=seed)), ITEM_STATS)
    m = lgb.LGBMClassifier(n_estimators=150, verbose=-1, random_state=seed).fit(tr[["pop_cum", "rating_mean_cum"]], tr.label)
    auc = roc_auc_score(f.label, m.predict_proba(f[["pop_cum", "rating_mean_cum"]])[:, 1])
    print(f"E1 · sin features del futuro: {ok_ts} · AUC val (PIT) = {auc:.3f}")
    return ok_ts


try:
    check_E1()
except NotImplementedError:
    print("⏳ E1 pendiente")

## Paso 2 · Candidatos y ranker LambdaMART (E2)

1. `build_candidates(hist, users, t_cut, labels=None)` → DataFrame con `user_id, item_id` y las columnas de `RANK_FEATS`
   usando **solo información anterior a `t_cut`** (perfil del usuario, géneros, features de ítem as-of `t_cut`).
2. Entrena un `lgb.LGBMRanker(objective="lambdarank")` agrupando por usuario (`group=`) con etiquetas de `train_b`.
3. Evalúa NDCG@10 en `test` (historial hasta `t_b`) frente al orden del retrieval.

💡 Pista: `affinity = GENRE_MAT[ids] @ perfil_generos_usuario / ‖GENRE_MAT[ids]‖`; el perfil es la media normalizada de los géneros vistos.

In [ ]:
def build_candidates(hist: dict, users: list, t_cut: float, k: int = 100, labels: dict | None = None) -> pd.DataFrame:
    # TODO: retrieval top-k + features (RANK_FEATS) as-of t_cut (+ columna label si labels no es None)
    raise NotImplementedError


def train_ranker(cands: pd.DataFrame):
    # TODO: LGBMRanker lambdarank con group por usuario (ordena por user_id antes)
    raise NotImplementedError


def check_E2(ranker, build=None):
    build = build or build_candidates
    hist_b = ratings[ratings.ts < t_b].groupby("user_id").item_id.apply(list).to_dict()
    test_pos = test.groupby("user_id").item_id.apply(set).to_dict()
    users = [u for u in test_pos if u in hist_b][:500]
    c = build(hist_b, users, t_b)
    c["score"] = ranker.predict(c[RANK_FEATS])
    nd_r = np.mean([ndcg_at_k(list(g.sort_values("retr_rank").item_id), test_pos[u]) for u, g in c.groupby("user_id")])
    nd_m = np.mean([ndcg_at_k(list(g.sort_values("score", ascending=False).item_id), test_pos[u]) for u, g in c.groupby("user_id")])
    print(f"E2 · NDCG@10 retrieval={nd_r:.4f} ranker={nd_m:.4f} ratio={nd_m / max(nd_r, 1e-9):.2f} (objetivo ≥ 1,10)")
    return nd_m / max(nd_r, 1e-9)


try:
    lab_b = train_b.groupby("user_id").item_id.apply(set).to_dict()
    cands = build_candidates(hist_a, [u for u in lab_b if u in hist_a], t_a, labels=lab_b)
    check_E2(train_ranker(cands))
except NotImplementedError:
    print("⏳ E2 pendiente")

## Paso 3 · El servicio (E3)

Completa la plantilla. Requisitos: carga de artefactos al arrancar (no por petición), caché con TTL e invalidación en
`/event`, fallback de popularidad, `/metrics` en texto Prometheus, y **log de features servidas** (un JSON por línea
en `logs/served_features.jsonl` con `user_id`, `item_id` y las 7 features) cuando `LOG_FEATURES=1`.

In [ ]:
%%writefile service_template.py
import json, os, time
import numpy as np
from fastapi import FastAPI, Response

app = FastAPI(title="CineMatch (plantilla)")
# TODO: cargar artefactos (índice FAISS, embeddings, ranker, features de usuario/ítem, historiales, popularidad)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/recommend/{user_id}")
def recommend(user_id: int, k: int = 10, use_cache: bool = True):
    # TODO: caché → historial → retrieval → features → ranker → diversidad → log de features → respuesta
    raise NotImplementedError


@app.post("/event")
def event(payload: dict):
    # TODO: actualizar historial + invalidar caché
    raise NotImplementedError


@app.get("/metrics")
def metrics():
    # TODO: contadores en formato Prometheus
    return Response("", media_type="text/plain")

## Paso 4 · Test de carga (E4) y paso 5 · Paridad de features (E5)

- `load_test(url, users, concurrency, n)` → dict con `qps`, `p50`, `p95`, `p99` medidos **desde el cliente**.
- `parity_check(log_path)` → fracción de celdas (fila × feature) en las que el log del servicio coincide con
  `build_candidates` recalculado offline para los mismos `(user_id, item_id)` (tolerancia 1e-5).

In [ ]:
def load_test(url: str, users: list, concurrency: int, n: int = 400, cache: bool = False) -> dict:
    # TODO
    raise NotImplementedError


def parity_check(log_path: str) -> float:
    # TODO
    raise NotImplementedError

## Paso 6 · Decisiones de arquitectura (E6)

✍️ Escribe aquí (markdown) qué piezas del servicio moverías a `reference_stack/` en producción real (Redis, Feast con
online store Redis, Redpanda/Kafka para `/event`, Triton para un ranker neuronal, Prometheus/Grafana) y qué SLO/alertas definirías.

---

# ⛔ SPOILER — intenta resolverlo primero

A partir de aquí está la **solución de referencia** completa y ejecutable.

In [ ]:
def pit_item_features(examples: pd.DataFrame, item_stats: pd.DataFrame) -> pd.DataFrame:
    st = item_stats.rename(columns={"event_timestamp": "feature_ts"}).sort_values("feature_ts")
    ex = examples.sort_values("event_timestamp")
    out = pd.merge_asof(ex, st, left_on="event_timestamp", right_on="feature_ts", by="item_id", direction="backward")
    return out.fillna({"pop_cum": 0, "rating_mean_cum": 3.0})


check_E1()

In [ ]:
def user_profile(df):
    g = df.groupby("user_id")
    prof = pd.DataFrame({"user_n": g.size().astype(float), "user_mean": g.rating.mean()})
    gen = pd.DataFrame(GENRE_MAT[df.item_id.values], index=df.user_id.values).groupby(level=0).mean()
    gen = gen.div(np.linalg.norm(gen.values, axis=1) + 1e-8, axis=0)
    return prof, gen


def item_feats_asof(t_cut):
    snap = ITEM_STATS[ITEM_STATS.event_timestamp <= pd.Timestamp(t_cut, unit="s", tz="UTC")]
    last = snap.groupby("item_id").tail(1).set_index("item_id")[["pop_cum", "rating_mean_cum"]]
    return last.reindex(range(N_ITEMS)).fillna({"pop_cum": 0, "rating_mean_cum": 3.0}).astype(float)


def features_for(u, ids, sc, prof, gen, itf):
    """ÚNICA implementación de las features: la usan el builder offline y (copiada al artefacto) el servicio."""
    aff = GENRE_MAT[ids] @ gen.loc[u].values / (np.linalg.norm(GENRE_MAT[ids], axis=1) + 1e-8)
    return pd.DataFrame({"user_id": u, "item_id": ids, "retr_score": sc.astype(float), "retr_rank": np.arange(len(ids), dtype=float),
                         "pop_cum": itf.pop_cum.values[ids], "rating_mean_cum": itf.rating_mean_cum.values[ids],
                         "genre_affinity": aff.astype(float), "user_n": prof.loc[u, "user_n"], "user_mean": prof.loc[u, "user_mean"]})


def build_candidates(hist, users, t_cut, k=100, labels=None):
    prof, gen = user_profile(ratings[ratings.ts < t_cut]); itf = item_feats_asof(t_cut)
    rows = []
    for u in users:
        ids, sc = retrieve(hist[u], k)
        if len(ids):
            rows.append(features_for(u, ids, sc, prof, gen, itf))
    out = pd.concat(rows, ignore_index=True)
    if labels is not None:
        out["label"] = [int(i in labels.get(u, ())) for u, i in zip(out.user_id, out.item_id)]
    return out


def train_ranker(cands):
    c = cands[cands.groupby("user_id").label.transform("max") > 0].sort_values("user_id")
    rk = lgb.LGBMRanker(objective="lambdarank", n_estimators=300, learning_rate=0.05, num_leaves=31,
                        min_child_samples=20, verbose=-1, random_state=seed)
    rk.fit(c[RANK_FEATS], c.label, group=c.groupby("user_id").size().values)
    return rk


lab_b = train_b.groupby("user_id").item_id.apply(set).to_dict()
cands = build_candidates(hist_a, [u for u in lab_b if u in hist_a], t_a, labels=lab_b)
RANKER = train_ranker(cands)
ratio = check_E2(RANKER)

In [ ]:
# Publicar artefactos as-of t_b (lo que haría el pipeline offline cada noche)
hist_b = ratings[ratings.ts < t_b].groupby("user_id").item_id.apply(list).to_dict()
prof_b, gen_b = user_profile(ratings[ratings.ts < t_b]); itf_b = item_feats_asof(t_b)
faiss.write_index(INDEX, f"{ART_DIR}/items.faiss")
np.save(f"{ART_DIR}/item_emb.npy", ITEM_EMB); np.save(f"{ART_DIR}/genre_mat.npy", GENRE_MAT)
RANKER.booster_.save_model(f"{ART_DIR}/ranker.txt")
itf_b.to_parquet(f"{ART_DIR}/item_feats.parquet"); prof_b.to_parquet(f"{ART_DIR}/user_prof.parquet")
gen_b.columns = [str(c) for c in gen_b.columns]; gen_b.to_parquet(f"{ART_DIR}/user_genre.parquet")
json.dump({str(u): h[-50:] for u, h in hist_b.items()}, open(f"{ART_DIR}/user_hist.json", "w"))
json.dump({"popular": ratings[ratings.ts < t_b].item_id.value_counts().index[:200].tolist(),
           "rank_feats": RANK_FEATS, "model_version": "ranker-lambdamart-v1"}, open(f"{ART_DIR}/meta.json", "w"))

In [ ]:
%%writefile cinematch_service.py
"""CineMatch · servicio de recomendación (solución de referencia del proyecto 16)."""
import json, os, threading, time
import faiss, fakeredis, lightgbm as lgb, numpy as np, pandas as pd
from fastapi import FastAPI, Response
from pydantic import BaseModel

ART = os.environ.get("ART_DIR", "artifacts"); LOG = os.environ.get("LOG_FEATURES", "0") == "1"
INDEX = faiss.read_index(f"{ART}/items.faiss"); INDEX.hnsw.efSearch = 64
EMB, GENRE = np.load(f"{ART}/item_emb.npy"), np.load(f"{ART}/genre_mat.npy")
GNORM = np.linalg.norm(GENRE, axis=1) + 1e-8
RANKER = lgb.Booster(model_file=f"{ART}/ranker.txt")
ITF = pd.read_parquet(f"{ART}/item_feats.parquet"); POP, RMEAN = ITF.pop_cum.to_numpy(), ITF.rating_mean_cum.to_numpy()
PROF = pd.read_parquet(f"{ART}/user_prof.parquet"); UGEN = pd.read_parquet(f"{ART}/user_genre.parquet")
PROF_D = {int(u): (float(r.user_n), float(r.user_mean)) for u, r in PROF.iterrows()}
UGEN_D = {int(u): row for u, row in zip(UGEN.index, UGEN.to_numpy())}
META = json.load(open(f"{ART}/meta.json")); FEATS = META["rank_feats"]
try:
    import redis
    R = redis.Redis(host=os.environ["REDIS_HOST"], decode_responses=True); R.ping()
except Exception:
    R = fakeredis.FakeRedis(decode_responses=True)
for u, h in json.load(open(f"{ART}/user_hist.json")).items():
    R.delete(f"hist:{u}"); R.rpush(f"hist:{u}", *h)
TTL = int(os.environ.get("CACHE_TTL", "60"))
BUCKETS = [5, 10, 20, 40, 80, 160, float("inf")]
M = {"requests_total": 0, "cache_hits_total": 0, "fallback_total": 0, "errors_total": 0}
HIST = [0] * len(BUCKETS); LAT_SUM = [0.0]; LOCK = threading.Lock()
os.makedirs("logs", exist_ok=True); LOG_F = open("logs/served_features.jsonl", "a") if LOG else None
app = FastAPI(title="CineMatch Recommender", version=META["model_version"])


class Event(BaseModel):
    user_id: int
    item_id: int
    event_type: str = "play"


def observe(ms):
    with LOCK:
        LAT_SUM[0] += ms
        for j, b in enumerate(BUCKETS):
            if ms <= b:
                HIST[j] += 1


def diversify(items, k, max_per_genre=3):
    out, cnt = [], {}
    for i in items:
        g = int(np.argmax(GENRE[i][1:])) + 1
        if cnt.get(g, 0) < max_per_genre:
            out.append(int(i)); cnt[g] = cnt.get(g, 0) + 1
        if len(out) == k:
            break
    return out


@app.get("/health")
def health():
    return {"status": "ok", "model_version": META["model_version"]}


@app.get("/recommend/{user_id}")
def recommend(user_id: int, k: int = 10, use_cache: bool = True):
    t0 = time.perf_counter(); M["requests_total"] += 1
    key = f"recs:{user_id}:{k}"
    if use_cache and (hit := R.get(key)):
        M["cache_hits_total"] += 1; observe(1e3 * (time.perf_counter() - t0))
        return {**json.loads(hit), "cache": True}
    hist = [int(x) for x in R.lrange(f"hist:{user_id}", 0, -1)]
    if not hist or user_id not in PROF_D:
        M["fallback_total"] += 1
        seen = set(hist); items = [i for i in META["popular"] if i not in seen][:k]
        observe(1e3 * (time.perf_counter() - t0))
        return {"user_id": user_id, "items": items, "strategy": "popularity_fallback", "cache": False}
    v = EMB[hist[-50:]].mean(0); v = (v / (np.linalg.norm(v) + 1e-8)).astype("float32")
    sc, ids = INDEX.search(v[None], 100 + len(hist))
    seen = set(hist); mask = np.array([(i not in seen) and i > 0 for i in ids[0]])
    ids, sc = ids[0][mask][:100], sc[0][mask][:100]
    n_u, mean_u = PROF_D[user_id]
    aff = GENRE[ids] @ UGEN_D[user_id] / GNORM[ids]
    X = np.column_stack([sc.astype(float), np.arange(len(ids), dtype=float), POP[ids], RMEAN[ids], aff,
                         np.full(len(ids), n_u), np.full(len(ids), mean_u)])
    order = ids[np.argsort(-RANKER.predict(X))]
    items = diversify(order, k)
    if LOG_F:
        for i, row in zip(ids, X):
            LOG_F.write(json.dumps({"user_id": user_id, "item_id": int(i), **dict(zip(FEATS, map(float, row)))}) + "\n")
        LOG_F.flush()
    resp = {"user_id": user_id, "items": items, "strategy": "two_stage", "model_version": META["model_version"]}
    R.setex(key, TTL, json.dumps(resp))
    observe(1e3 * (time.perf_counter() - t0))
    return {**resp, "cache": False}


@app.post("/event")
def event(e: Event):
    R.rpush(f"hist:{e.user_id}", e.item_id); R.ltrim(f"hist:{e.user_id}", -50, -1)
    for key in R.scan_iter(f"recs:{e.user_id}:*"):
        R.delete(key)
    return {"ok": True}


@app.get("/metrics")
def metrics():
    lines = [f"# TYPE cinematch_{k} counter\ncinematch_{k} {v}" for k, v in M.items()]
    lines.append("# TYPE cinematch_latency_ms histogram")
    for b, c in zip(BUCKETS, HIST):
        lines.append(f'cinematch_latency_ms_bucket{{le="{"+Inf" if b == float("inf") else b}"}} {c}')
    lines += [f"cinematch_latency_ms_sum {LAT_SUM[0]}", f"cinematch_latency_ms_count {HIST[-1]}"]
    return Response("\n".join(lines) + "\n", media_type="text/plain")

In [ ]:
def start_service(port=8001, log_features=True):
    if os.path.exists("logs/served_features.jsonl"):
        os.remove("logs/served_features.jsonl")
    env = {**os.environ, "ART_DIR": ART_DIR, "LOG_FEATURES": "1" if log_features else "0"}
    p = subprocess.Popen([sys.executable, "-m", "uvicorn", "cinematch_service:app", "--port", str(port), "--log-level", "warning"], env=env)
    for _ in range(120):
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=0.5).ok:
                return p
        except requests.exceptions.RequestException:
            time.sleep(0.5)
    p.kill(); raise RuntimeError("no arranca")


URL = "http://127.0.0.1:8001"
svc = start_service()
test_pos = test.groupby("user_id").item_id.apply(set).to_dict()
users_t = [u for u in test_pos if u in hist_b]
u0 = users_t[0]
a = requests.get(f"{URL}/recommend/{u0}").json(); b = requests.get(f"{URL}/recommend/{u0}").json()
requests.post(f"{URL}/event", json={"user_id": u0, "item_id": a["items"][0]})
c = requests.get(f"{URL}/recommend/{u0}").json()
new = requests.get(f"{URL}/recommend/123456789").json()
E3 = {"cache_2a": b["cache"], "invalidada_tras_evento": not c["cache"], "item_visto_excluido": a["items"][0] not in c["items"],
      "fallback_usuario_nuevo": new["strategy"] == "popularity_fallback" and len(new["items"]) == 10,
      "metrics_ok": "cinematch_requests_total" in requests.get(f"{URL}/metrics").text}
print("E3:", E3, "→", "✅" if all(E3.values()) else "❌")

In [ ]:
def load_test(url, users, concurrency, n=400, cache=False):
    sess = requests.Session()
    def one(u):
        t = time.perf_counter(); r_ = sess.get(f"{url}/recommend/{u}", params={"use_cache": cache}); r_.raise_for_status()
        return 1e3 * (time.perf_counter() - t)
    us = np.random.default_rng(seed).choice(users, n); t0 = time.perf_counter()
    with ThreadPoolExecutor(concurrency) as ex:
        lat = np.array(list(ex.map(one, us)))
    return {"concurrencia": concurrency, "qps": n / (time.perf_counter() - t0),
            **{f"p{q}": np.percentile(lat, q) for q in (50, 95, 99)}}


for _ in range(30):                       # warm-up
    requests.get(f"{URL}/recommend/{users_t[_]}", params={"use_cache": False})
lt = pd.DataFrame([load_test(URL, users_t, c) for c in [1, 2, 4, 8, 16]])
display(lt.round(1))
p99_8 = float(lt.loc[lt.concurrencia == 8, "p99"].iloc[0])
print(f"E4 · p99 con 8 concurrentes = {p99_8:.1f} ms → {'✅' if p99_8 <= 80 else '❌ (optimiza o añade workers)'}")
fig, ax = plt.subplots(figsize=(8, 4))
for col in ["p50", "p95", "p99"]:
    ax.plot(lt.concurrencia, lt[col], "o-", label=col)
ax.axhline(80, ls="--", color="r", label="SLO"); ax.set_xlabel("concurrencia"); ax.set_ylabel("ms")
ax.set_title("Proyecto 16 · latencia desde el cliente"); ax.legend(); plt.show()

In [ ]:
def parity_check(log_path="logs/served_features.jsonl"):
    logs = pd.read_json(log_path, lines=True).drop_duplicates(["user_id", "item_id"], keep="first")
    logs = logs[logs.user_id != u0]                      # u0 recibió un evento nearline: su historial cambió
    users = logs.user_id.unique()[:200]
    logs = logs[logs.user_id.isin(users)]
    prof, gen = user_profile(ratings[ratings.ts < t_b]); itf = item_feats_asof(t_b)
    off = []
    for u, g in logs.groupby("user_id"):
        ids, sc = retrieve(hist_b[u], 100)                 # historial del batch (sin eventos nearline)
        off.append(features_for(u, ids, sc, prof, gen, itf))
    off = pd.concat(off)
    j = logs.merge(off, on=["user_id", "item_id"], suffixes=("_srv", "_off"))
    match = np.mean([np.isclose(j[f + "_srv"], j[f + "_off"], atol=1e-5).mean() for f in RANK_FEATS])
    print(f"E5 · filas comparadas: {len(j)} · paridad = {match:.4%}")
    return match


parity = parity_check()
svc.terminate()

⚠️ Nota sobre E5: excluimos de la comparación al usuario `u0` al que le enviamos un evento (su historial cambió *nearline*,
así que sus features legítimamente difieren del batch). En producción, el *log de features servidas* es precisamente lo
que permite entrenar con la verdad de serving en vez de recalcularla.

**E6 (ejemplo de respuesta):** Redis gestionado para historial y caché (TTL 60 s + invalidación por evento); `/event`
publica en Redpanda (topic `cinematch.events`, particionado por `user_id`) y un consumidor nearline actualiza Redis;
features de usuario/ítem en Feast con online store Redis materializado cada hora; el ranker neuronal (si sustituye a
LightGBM) en Triton con *dynamic batching*; Prometheus raspa `/metrics`, alerta si p99 > 80 ms durante 5 min o si
`fallback_total/requests_total` > 5 %. Ver `reference_stack/`.

## 🚀 Retos extra (nivel experto)

1. Levanta el `reference_stack/docker-compose.yml` en tu máquina (Redis + Redpanda + Prometheus + Grafana) y apunta el servicio a Redis real con `REDIS_HOST`.
2. Sustituye el ranker LightGBM por un MLP exportado a ONNX (lección §10) y sírvelo con Triton (`reference_stack/triton/`); compara p99.
3. Implementa *hedged requests* en el cliente del load test y mide el efecto en p99.
4. Haz el endpoint asíncrono y mueve la inferencia a un `ThreadPoolExecutor` / `run_in_threadpool`; compara con `--workers 4`.
5. Sustituye las features de ítem por Feast (`get_online_features`) y mide cuánto añade a la latencia.
6. Añade un *circuit breaker*: si el ranker supera 30 ms, devuelve el orden del retrieval (degradación elegante) y cuéntalo en `/metrics`.

## 🤔 Reflexión (conecta con MLOps)

- ¿Qué versionarías juntos para que un rollback sea atómico (modelo, índice, features, `meta.json`)?
- ¿Qué alertas pondrías sobre `/metrics` y cuáles requieren datos de negocio (CTR, play rate) que el servicio no ve?
- Si mañana el batch nocturno de features falla, ¿qué ve el modelo? ¿Cómo lo detectas antes que el usuario?
- ¿Cómo harías *shadow traffic* para probar un ranker nuevo con tráfico real sin afectar a nadie? (Módulo 17.)